# RADIOLOGY - REPORTING HARNESS

## 1. Requirements

In [1]:
!pip install -q -U "trl>=0.28.0" "transformers>=4.57.0" "peft>=0.17.0" "accelerate>=1.10.0" "datasets>=3.6.0" "bitsandbytes>=0.46.0"

## 2. Imports and config

In [2]:
import os, re, json, math, unicodedata, warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")

# ----------------------------- paths -----------------------------
MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_CSV  = "/content/train.csv"
TEST_CSV   = "/content/test.csv"
REPORT_COL = "report"
VAL_FRACTION = 0.15

OUTPUT_DIR  = "/content/outputs"
ADAPTER_DIR = "/content/qwen2.5-3b-radiology-lora"
PREDICTIONS_CSV = "/content/validation_predictions.csv"
SCORES_CSV      = "/content/validation_scores.csv"
SUBMISSION_CSV  = "/content/submission.csv"

# --------------------------- generation --------------------------
MAX_NEW_TOKENS       = 1280   # first pass
RETRY_MAX_NEW_TOKENS = 2048   # for outputs that did not finish in the first pass
GEN_BATCH_SIZE       = 4      # lower to 2 or 1 if you hit CUDA OOM

SEED = 42
set_seed(SEED)

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime > Change runtime type > T4)."
for p in [TRAIN_CSV, TEST_CSV]:
    if not os.path.exists(p):
        print("MISSING FILE:", p)

## 3. Prompt (single source of truth for training and inference)

In [3]:
SYSTEM_PROMPT = (
    "You are an expert radiology report editor. "
    "Generate the final report from the supplied template and dictation. "
    "Preserve template structure and wording where appropriate. "
    "Do not invent findings."
)

def _get(x, key):
    v = x.get(key, "")
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    return str(v)

def build_user_prompt(x):
    return "\n".join([
        "Generate the final radiology report from the supplied template and dictation.",
        "",
        "Requirements:",
        "- Incorporate all relevant dictated findings accurately.",
        "- Preserve the template's structure and wording where appropriate.",
        "- Do not invent findings.",
        "- Produce exactly two top-level sections:",
        "  FINDINGS:",
        "  IMPRESSION:",
        "- Return only the final report.",
        "",
        "MODALITY:", _get(x, "modality"), "",
        "BODY PART:", _get(x, "body_part"), "",
        "STUDY DESCRIPTION:", _get(x, "study_description"), "",
        "PATIENT AGE BAND:", _get(x, "patient_age_band"), "",
        "PATIENT SEX:", _get(x, "patient_sex"), "",
        "TEMPLATE:", _get(x, "template_content"), "",
        "DICTATION:", _get(x, "dictation"),
    ]).strip()

def build_prompt_messages(x):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(x)},
    ]

## 4. Tokenizer, 4-bit model, QLoRA

In [4]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
EOS_IDS = sorted({i for i in [IM_END_ID, tok.eos_token_id, tok.pad_token_id] if i is not None})

def make_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=make_bnb_config(),
    device_map="auto",
    dtype=torch.float16,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

def force_trainable_fp32(m):
    # fp16 AMP needs fp32 trainable params; bf16 grads crash the grad scaler
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
    from collections import Counter
    print("trainable param dtypes:", dict(Counter(str(p.dtype) for p in m.parameters() if p.requires_grad)))
    print("all param dtypes:      ", dict(Counter(str(p.dtype) for p in m.parameters())))

force_trainable_fp32(model)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable param dtypes: {'torch.float32': 504}
all param dtypes:       {'torch.float32': 686, 'torch.uint8': 252}
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 5. Data (prompt/completion format, loss on the report only)

In [5]:
from datasets import Dataset

# ---- CSV loader that survives unbalanced quotes ----
import csv, io, re

_UUID_START = re.compile(
    r'(?m)^"?[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"?,')

def load_csv_robust(path, allow_drop=True):
    """pd.read_csv, with a record-level repair if the file has unbalanced quotes."""
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"pd.read_csv failed ({type(e).__name__}: {str(e)[:120]})\n-> repairing record by record")

    raw = open(path, encoding="utf-8-sig", newline="").read()
    nl = raw.find("\n")
    header = next(csv.reader([raw[:nl].rstrip("\r")]))
    body = raw[nl + 1:]
    n = len(header)
    t_idx = header.index("template_content") if "template_content" in header else n - 3

    starts = [m.start() for m in _UUID_START.finditer(body)]
    if not starts:
        raise ValueError("Could not find rows starting with a UUID case_id; the file needs manual inspection.")
    starts.append(len(body))

    rows, repaired, dropped = [], [], []
    for k, (a, b) in enumerate(zip(starts[:-1], starts[1:])):
        rec = body[a:b].rstrip("\r\n")
        try:                                   # 1) strict CSV parse of this one record
            parsed = list(csv.reader(io.StringIO(rec), strict=True))
            if len(parsed) == 1 and len(parsed[0]) == n:
                rows.append(parsed[0]); continue
        except csv.Error:
            pass
        # 2) manual split: short leading fields by comma, long text fields by the "," joiner
        lead_and_rest = rec.split(",", t_idx)
        fixed = None
        if len(lead_and_rest) == t_idx + 1:
            rest = lead_and_rest[t_idx].strip()
            rest = rest[1:] if rest.startswith('"') else rest
            rest = rest[:-1] if rest.endswith('"') else rest
            parts = rest.split('","')
            if len(parts) == n - t_idx:
                fixed = [x.strip().strip('"') for x in lead_and_rest[:t_idx]] + [p.replace('""', '"') for p in parts]
        if fixed:
            rows.append(fixed); repaired.append((k, rec[:160].replace("\n", " ")))
        else:
            dropped.append((k, rec[:160].replace("\n", " ")))

    print(f"records found: {len(starts) - 1} | parsed: {len(rows)} (repaired {len(repaired)}) | dropped: {len(dropped)}")
    for k, snip in repaired[:3]:
        print(f"  repaired record #{k}: {snip}")
    for k, snip in dropped[:3]:
        print(f"  DROPPED record #{k}: {snip}")
    if dropped and not allow_drop:
        raise ValueError(f"{len(dropped)} records could not be parsed in {path}")
    return pd.DataFrame(rows, columns=header)

INPUT_COLS = ["case_id", "modality", "body_part", "study_description",
              "patient_age_band", "patient_sex", "template_content", "dictation"]

full_df = load_csv_robust(TRAIN_CSV)
print("train.csv shape:", full_df.shape)
print("columns:", full_df.columns.tolist())

needed = [c for c in INPUT_COLS if c != "case_id"] + [REPORT_COL]
missing = [c for c in needed if c not in full_df.columns]
assert not missing, f"train.csv is missing columns {missing}. Fix REPORT_COL / column names above."

if "case_id" not in full_df.columns:
    full_df["case_id"] = range(len(full_df))
full_df = full_df.dropna(subset=[REPORT_COL]).reset_index(drop=True)

# ---- random train/val split from the single CSV ----
perm = np.random.RandomState(SEED).permutation(len(full_df))
n_val = max(1, int(len(full_df) * VAL_FRACTION))
val_df   = full_df.iloc[perm[:n_val]].reset_index(drop=True)
train_df = full_df.iloc[perm[n_val:]].reset_index(drop=True)

def to_records(df):
    recs = []
    for _, r in df.iterrows():
        inp = {c: ("" if pd.isna(r[c]) else str(r[c])) for c in INPUT_COLS}
        recs.append({"input": inp, "output": {"report": str(r[REPORT_COL])}})
    return recs

raw_train = Dataset.from_list(to_records(train_df))
raw_val   = Dataset.from_list(to_records(val_df))
print("train:", len(raw_train), "| val:", len(raw_val))

def to_prompt_completion(ex):
    return {
        "prompt": build_prompt_messages(ex["input"]),
        "completion": [{"role": "assistant", "content": str(ex["output"].get("report", ""))}],
    }

train_ds = raw_train.map(to_prompt_completion, remove_columns=raw_train.column_names)
val_ds   = raw_val.map(to_prompt_completion, remove_columns=raw_val.column_names)

# ---- token lengths of full conversations (prompt + report), no truncation ----
def full_len(ex):
    text = tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
    return len(tok(text, add_special_tokens=False)["input_ids"])

lens = np.array([full_len(e) for e in list(train_ds) + list(val_ds)])
print(f"min {lens.min()} | mean {lens.mean():.0f} | p95 {np.percentile(lens, 95):.0f} | max {lens.max()}")

LEN_CAP = 2560   # memory safety cap for a T4
MAX_LEN = int(min(LEN_CAP, math.ceil(lens.max() / 128) * 128))
print("MAX_LEN for training:", MAX_LEN)
n_over = int((lens > MAX_LEN).sum())
if n_over:
    print(f"WARNING: {n_over} examples exceed MAX_LEN and will be truncated.")

# ---- sanity check: what the model sees ----
ex0 = train_ds[0]
print("\n--- PROMPT (tail) ---\n", ex0["prompt"][-1]["content"][-300:])
print("\n--- COMPLETION (head) ---\n", ex0["completion"][0]["content"][:300])

train.csv shape: (636, 9)
columns: ['case_id', 'modality', 'body_part', 'study_description', 'patient_age_band', 'patient_sex', 'template_content', 'dictation', 'report']
train: 541 | val: 95


Map:   0%|          | 0/541 [00:00<?, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]

min 275 | mean 678 | p95 1401 | max 2106
MAX_LEN for training: 2176

--- PROMPT (tail) ---
 TE:
FINDINGS:
BONES: No acute fracture or focal osseous lesion.
JOINTS: No dislocation. The joint spaces are normal.
SOFT TISSUES: The soft tissues are unremarkable.

IMPRESSION:
No acute osseous abnormality.

DICTATION:
mild lateral swelling of the 5th digit at the area of metacarphophlangeal joint

--- COMPLETION (head) ---
 FINDINGS:
BONES: No acute fracture or focal osseous lesion.

JOINTS: No dislocation. The joint spaces are normal.

SOFT TISSUES: There is mild lateral swelling of the 5th digit at the area of the metatarsophalangeal joint. The remaining soft tissues are unremarkable.

IMPRESSION:
1. Mild lateral sof


## 6. Train

In [6]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # training
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",

    fp16=False,
    bf16=False,

    # memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # sequence / loss
    max_length=MAX_LEN,
    packing=False,
    completion_only_loss=True,      # loss on the assistant report only

    # evaluation + checkpointing
    eval_strategy="steps",
    eval_steps=50,
    prediction_loss_only=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
)

force_trainable_fp32(trainer.model)
trainer.train()

Tokenizing train dataset:   0%|          | 0/541 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/541 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/541 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/541 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

trainable param dtypes: {'torch.float32': 504}
all param dtypes:       {'torch.float32': 686, 'torch.uint8': 252}


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.201442,0.201670,0.216631,0.943038,273199.000000
100,0.126777,0.169299,0.156558,0.950024,548072.000000
150,0.095752,0.164695,0.131703,0.952219,819109.000000
200,0.097311,0.163520,0.117008,0.952903,1090398.000000
204,0.097311,0.163519,0.117002,0.952968,1115082.000000


TrainOutput(global_step=204, training_loss=0.15910759641259326, metrics={'train_runtime': 3336.7282, 'train_samples_per_second': 0.486, 'train_steps_per_second': 0.061, 'total_flos': 1.8764871940325376e+16, 'train_loss': 0.15910759641259326, 'epoch': 3.0})

## 7. Save adapter

In [7]:
trainer.save_model(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
print("Saved adapter to:", ADAPTER_DIR)

Saved adapter to: /content/qwen2.5-3b-radiology-lora


In [15]:
from huggingface_hub import HfApi, login, create_repo, notebook_login

HF_REPO_ID = "caffeic/qwen2.5-3b-radiology-lora"
PRIVATE = True

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    notebook_login()          # fallback: opens a token prompt

assert os.path.isdir(ADAPTER_DIR), "Run the 'Save adapter' cell first."

create_repo(HF_REPO_ID, private=PRIVATE, exist_ok=True)

HfApi().upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="QLoRA adapter: Qwen2.5-3B-Instruct radiology report editor",
)
print("Uploaded:", f"https://huggingface.co/{HF_REPO_ID}")

Uploaded: https://huggingface.co/caffeic/qwen2.5-3b-radiology-lora


## 8. Scorer (local approximation of the competition metric)
This is your own re-implementation, so compare it with the official metric definition. Fixes: signed numbers and decimals are now preserved, mixed-case field labels (like `Bones:`) are parsed.

In [8]:
UNITS = {"millimeters": "mm", "millimeter": "mm", "centimeters": "cm", "centimeter": "cm"}

FUNCTION_WORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "at", "for", "with", "without",
    "from", "by", "as", "is", "are", "was", "were", "be", "been", "being", "this", "that",
    "these", "those", "there", "it", "its", "than", "into", "over", "under", "within",
}
CRITICAL_WORDS = {
    "no", "not", "without", "negative", "absent", "right", "left", "bilateral",
    "mild", "moderate", "severe", "marked", "minimal", "trace", "small", "large",
    "acute", "chronic", "subacute", "new", "stable", "fracture", "dislocation",
    "tear", "rupture", "effusion", "pneumothorax", "pneumonia", "mass", "lesion",
    "stenosis", "occlusion", "thrombosis", "edema", "swelling",
}
UNIT_WORDS = {"mm", "cm"}

PENALIZE_EXTRA_FIELDS = False   # my addition: also penalize invented labeled fields (off = same as before)

NUM_RE = re.compile(r"[+-]?\d+(?:\.\d+)?")

def normalize_for_score(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text)).lower()

    for src, dst in UNITS.items():
        text = re.sub(rf"\b{re.escape(src)}\b", dst, text)

    text = re.sub(r"(?m)^\s*(?:\d+[\.\)]|[-*•])\s+", "", text)   # list markers
    text = re.sub(r"(?<=[a-z])-(?=[a-z])", "", text)              # air-space -> airspace
    text = re.sub(r"(?<=[a-z])(?=\d)", " ", text)                 # l5 -> l 5
    text = re.sub(r"(?<=\d)(?=[a-z])", " ", text)                 # 5mm -> 5 mm

    # protect numbers (signed + decimal) with LOWERCASE placeholders, so the
    # punctuation filter below cannot destroy them
    protected = []
    def protect(m):
        protected.append(m.group(0))
        return f" zznum{len(protected) - 1}zz "
    text = re.sub(r"(?<![\w.])[+-]?\d+(?:\.\d+)?", protect, text)

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"zznum(\d+)zz", lambda m: protected[int(m.group(1))], text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize_score(text):
    t = normalize_for_score(text)
    return t.split() if t else []

def token_weight(token):
    if NUM_RE.fullmatch(token):
        return 4.0
    if token in UNIT_WORDS or token in CRITICAL_WORDS:
        return 4.0
    if token in FUNCTION_WORDS:
        return 0.25
    return 2.0

def weighted_levenshtein(ref, pred):
    n, m = len(ref), len(pred)
    if n == 0 and m == 0:
        return 0.0
    rw = [token_weight(t) for t in ref]
    pw = [token_weight(t) for t in pred]

    prev = [0.0] * (m + 1)
    for j in range(1, m + 1):
        prev[j] = prev[j - 1] + pw[j - 1]
    for i in range(1, n + 1):
        cur = [prev[0] + rw[i - 1]] + [0.0] * m
        for j in range(1, m + 1):
            sub = prev[j - 1] + (0.0 if ref[i - 1] == pred[j - 1] else max(rw[i - 1], pw[j - 1]))
            cur[j] = min(prev[j] + rw[i - 1], cur[j - 1] + pw[j - 1], sub)
        prev = cur

    denom = max(sum(rw), sum(pw))
    return min(prev[m] / denom, 1.0) if denom else 0.0

def word_edit_score(reference, prediction):
    return weighted_levenshtein(tokenize_score(reference), tokenize_score(prediction))

def extract_report_sections(report):
    report = str(report)
    f = re.search(r"FINDINGS\s*:\s*(.*?)(?=\n\s*IMPRESSION\s*:|\Z)", report, flags=re.I | re.S)
    i = re.search(r"IMPRESSION\s*:\s*(.*)\Z", report, flags=re.I | re.S)
    return (f.group(1).strip() if f else ""), (i.group(1).strip() if i else "")

LABEL_RE = re.compile(r"^\s*([A-Za-z][A-Za-z0-9 /&(),.\-]{0,59}?)\s*:\s*(.*)$")

def parse_labeled_fields(text):
    fields, label, buf = {}, None, []
    def flush():
        if label is not None:
            content = " ".join(buf).strip()
            if content:
                fields[label] = content
    for line in (text or "").splitlines():
        m = LABEL_RE.match(line)
        if m and len(m.group(1).split()) <= 6:
            flush()
            label = re.sub(r"\s+", " ", m.group(1)).strip().upper()
            buf = [m.group(2).strip()] if m.group(2).strip() else []
        elif label is not None and line.strip():
            buf.append(line.strip())
    flush()
    return fields

def calculate_findings_score(template, reference, prediction):
    ref_f, _ = extract_report_sections(reference)
    pred_f, _ = extract_report_sections(prediction)
    tpl_f, _ = extract_report_sections(template)

    ref_fields = parse_labeled_fields(ref_f)
    pred_fields = parse_labeled_fields(pred_f)
    tpl_fields = parse_labeled_fields(tpl_f)

    if not ref_fields:
        return word_edit_score(ref_f, pred_f)

    num = den = 0.0
    for field, ref_text in ref_fields.items():
        changed = normalize_for_score(tpl_fields.get(field, "")) != normalize_for_score(ref_text)
        w = 3.0 if changed else 1.0
        num += w * word_edit_score(ref_text, pred_fields.get(field, ""))
        den += w

    for field in tpl_fields:
        if field not in ref_fields:
            num += word_edit_score("", pred_fields.get(field, ""))
            den += 1.0

    if PENALIZE_EXTRA_FIELDS:
        for field, text in pred_fields.items():
            if field not in ref_fields and field not in tpl_fields:
                num += word_edit_score("", text)
                den += 1.0

    return min(num / den, 1.0) if den else word_edit_score(ref_f, pred_f)

def calculate_impression_score(reference, prediction):
    _, ref_i = extract_report_sections(reference)
    _, pred_i = extract_report_sections(prediction)
    return word_edit_score(ref_i, pred_i)

def score_report(template, reference, prediction):
    f = calculate_findings_score(template, reference, prediction)
    i = calculate_impression_score(reference, prediction)
    return f, i, 0.65 * f + 0.35 * i      # lower is better

def evaluate(records, predictions):
    rows = []
    for k, (rec, pred) in enumerate(zip(records, predictions)):
        tpl = str(rec["input"].get("template_content", ""))
        ref = str(rec["output"].get("report", ""))
        f, i, res = score_report(tpl, ref, pred)
        rows.append({"case_id": rec["input"].get("case_id", k),
                     "findings_score": f, "impression_score": i, "res": res})
    return pd.DataFrame(rows)

def print_summary(name, df):
    print(f"{name:<22} RES {df['res'].mean():.4f} | FINDINGS {df['findings_score'].mean():.4f} | "
          f"IMPRESSION {df['impression_score'].mean():.4f} | exact {int(np.isclose(df['res'], 0).sum())}/{len(df)}")

## 9. Report cleanup (no invented text)

In [9]:
def clean_report(text):
    t = str(text).replace("<|im_end|>", "").strip()
    m = re.search(r"\bFINDINGS\s*:", t, flags=re.I)
    if m:
        t = t[m.start():]
    t = re.sub(r"^\s*FINDINGS\s*:", "FINDINGS:", t, flags=re.I)
    t = re.sub(r"\n\s*IMPRESSION\s*:", "\n\nIMPRESSION:", t, flags=re.I)
    return t.strip()

def finalize_report(raw_text, template):
    # Returns (report, status). Fallbacks use the TEMPLATE's own text and are counted,
    # so nothing is invented.
    t = clean_report(raw_text)
    has_f = re.search(r"FINDINGS\s*:", t, flags=re.I)
    has_i = re.search(r"IMPRESSION\s*:", t, flags=re.I)
    _, tpl_impression = extract_report_sections(template)

    if not has_f:
        return clean_report(template), "fallback_template"
    if not has_i:
        return (t + "\n\nIMPRESSION:\n" + tpl_impression).strip(), "fallback_template_impression"
    return t, "ok"

## 10. Batched generation

In [10]:
def render_prompt(x):
    return tok.apply_chat_template(build_prompt_messages(x), tokenize=False, add_generation_prompt=True)

def get_inference_model():
    if "trainer" in globals():
        m = trainer.model
    else:   # fresh runtime: base model in 4-bit + saved adapter
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=make_bnb_config(), device_map="auto", dtype=torch.float16)
        m = PeftModel.from_pretrained(base, ADAPTER_DIR)
    m.eval()
    m.config.use_cache = True
    return m

@torch.no_grad()
def _generate(prompts, max_new_tokens, batch_size):
    tok.padding_side = "left"          # required for batched generation
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    texts, finished = [None] * len(prompts), [False] * len(prompts)

    for s in tqdm(range(0, len(order), batch_size), desc="generating"):
        idx = order[s:s + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(infer_model.device)       # NO truncation
        out = infer_model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS, use_cache=True)
        new = out[:, enc["input_ids"].shape[1]:]
        for row, i in zip(new, idx):
            ids = row.tolist()
            finished[i] = any(e in ids for e in EOS_IDS)
            texts[i] = tok.decode(ids, skip_special_tokens=True).strip()
    tok.padding_side = "right"
    return texts, finished

def generate_reports(inputs, batch_size=GEN_BATCH_SIZE):
    prompts = [render_prompt(x) for x in inputs]
    plen = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
    print(f"prompt tokens: max {max(plen)} | mean {np.mean(plen):.0f}")

    texts, finished = _generate(prompts, MAX_NEW_TOKENS, batch_size)

    redo = [i for i, f in enumerate(finished) if not f]
    if redo:
        print(f"{len(redo)} outputs hit {MAX_NEW_TOKENS} tokens, retrying with {RETRY_MAX_NEW_TOKENS}")
        t2, f2 = _generate([prompts[i] for i in redo], RETRY_MAX_NEW_TOKENS, 1)
        for k, i in enumerate(redo):
            texts[i], finished[i] = t2[k], f2[k]
    print("unfinished after retry:", sum(1 for f in finished if not f))
    return texts, finished

## 11. Validation (with baselines)
Compare the fine-tuned model with the trivial **copy the template** baseline. If you cannot beat it, something is wrong.

In [11]:
val_records = [dict(r) for r in raw_val]
val_inputs  = [r["input"] for r in val_records]

# ---- baseline 1: copy the template unchanged ----
base_preds = [clean_report(str(x.get("template_content", ""))) for x in val_inputs]
df_copy = evaluate(val_records, base_preds)
print_summary("copy-template baseline", df_copy)

# ---- fine-tuned model ----
infer_model = get_inference_model()
torch.cuda.empty_cache()

raw_out, finished = generate_reports(val_inputs)
final = [finalize_report(r, x.get("template_content", "")) for r, x in zip(raw_out, val_inputs)]
val_preds  = [f[0] for f in final]
val_status = [f[1] for f in final]

df_model = evaluate(val_records, val_preds)
print()
print_summary("copy-template baseline", df_copy)
print_summary("fine-tuned model", df_model)
print("status counts:", pd.Series(val_status).value_counts().to_dict())

copy-template baseline RES 0.6380 | FINDINGS 0.5862 | IMPRESSION 0.7342 | exact 6/95
prompt tokens: max 1013 | mean 406


generating:   0%|          | 0/24 [00:00<?, ?it/s]

1 outputs hit 1280 tokens, retrying with 2048


generating:   0%|          | 0/1 [00:00<?, ?it/s]

unfinished after retry: 1

copy-template baseline RES 0.6380 | FINDINGS 0.5862 | IMPRESSION 0.7342 | exact 6/95
fine-tuned model       RES 0.2872 | FINDINGS 0.2564 | IMPRESSION 0.3444 | exact 11/95
status counts: {'ok': 94, 'fallback_template_impression': 1}


In [12]:
# ---- save predictions + scores, show worst cases ----
pred_df = pd.DataFrame({
    "case_id": df_model["case_id"],
    "modality": [x.get("modality", "") for x in val_inputs],
    "body_part": [x.get("body_part", "") for x in val_inputs],
    "template_content": [x.get("template_content", "") for x in val_inputs],
    "dictation": [x.get("dictation", "") for x in val_inputs],
    "reference_report": [r["output"].get("report", "") for r in val_records],
    "predicted_report": val_preds,
    "status": val_status,
    "findings_score": df_model["findings_score"],
    "impression_score": df_model["impression_score"],
    "res": df_model["res"],
})
pred_df.to_csv(PREDICTIONS_CSV, index=False, encoding="utf-8")
df_model.to_csv(SCORES_CSV, index=False, encoding="utf-8")

print(pred_df.sort_values("res", ascending=False)[["case_id", "status", "findings_score", "impression_score", "res"]].head(10).to_string(index=False))

# cases where the model is WORSE than copying the template
worse = (df_model["res"].values > df_copy["res"].values).sum()
print(f"\nmodel worse than copy-template on {worse}/{len(df_model)} cases")

                             case_id                       status  findings_score  impression_score      res
14862dc4-ba90-462d-9661-8f51a5a5fe1e fallback_template_impression        0.998197          0.926316 0.973038
5794f95d-ea96-44f9-b595-88f7aac2a190                           ok        0.603598          1.000000 0.742339
ca00b104-301f-453d-8880-8a100bfb8eb8                           ok        0.527594          0.982379 0.686769
831d8f42-8abf-40c9-a577-f626a6c2abaa                           ok        0.610257          0.740541 0.655856
92f479c4-ea89-464d-bf57-f7db69af0476                           ok        0.404494          1.000000 0.612921
9a18d29e-a76f-48c6-8e26-467371510a2d                           ok        0.467002          0.877358 0.610626
2db4817b-713b-4282-bf71-0579e9cc25c3                           ok        0.439419          0.907463 0.603234
e779d70f-d4f3-4ac9-b84d-4e97b640b1d1                           ok        0.436095          0.891156 0.595366
87d86f24-c4aa-4ac4-

In [13]:
# OPTIONAL: zero-shot baseline (adapter disabled). Slow, so it is off by default.
RUN_ZEROSHOT_BASELINE = False
if RUN_ZEROSHOT_BASELINE:
    with infer_model.disable_adapter():
        raw0, _ = generate_reports(val_inputs)
    zs = [finalize_report(r, x.get("template_content", ""))[0] for r, x in zip(raw0, val_inputs)]
    print_summary("zero-shot base model", evaluate(val_records, zs))

## 12. Test predictions and submission

In [14]:
test_df = pd.read_csv(TEST_CSV)
print("Test shape:", test_df.shape, "| columns:", test_df.columns.tolist())

def row_to_input(row):
    d = row.to_dict()
    return {k: ("" if (isinstance(v, float) and math.isnan(v)) else v) for k, v in d.items()}

test_inputs = [row_to_input(r) for _, r in test_df.iterrows()]

if "infer_model" not in globals():
    infer_model = get_inference_model()
torch.cuda.empty_cache()

raw_test, fin_test = generate_reports(test_inputs)
final_test = [finalize_report(r, x.get("template_content", "")) for r, x in zip(raw_test, test_inputs)]
test_preds  = [f[0] for f in final_test]
test_status = [f[1] for f in final_test]
print("status counts:", pd.Series(test_status).value_counts().to_dict())

submission = pd.DataFrame({"case_id": test_df["case_id"], "report": test_preds})

# ---- checks ----
assert len(submission) == len(test_df)
assert submission["case_id"].nunique() == len(submission), "Duplicate case_id"
assert set(submission["case_id"]) == set(test_df["case_id"]), "case_id mismatch"
assert submission["report"].notna().all()
assert submission["report"].str.strip().ne("").all()

bad = submission[~submission["report"].str.contains(r"FINDINGS\s*:", case=False, regex=True)
                 | ~submission["report"].str.contains(r"IMPRESSION\s*:", case=False, regex=True)]
print("reports missing FINDINGS/IMPRESSION:", len(bad))

import csv
submission.to_csv(SUBMISSION_CSV, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
pd.DataFrame({"case_id": test_df["case_id"], "status": test_status}).to_csv("/content/test_status.csv", index=False)

check = pd.read_csv(SUBMISSION_CSV)
print("Rows:", len(check), "| Columns:", check.columns.tolist())
print("\nFirst prediction:\n", check.iloc[0]["report"])

Test shape: (132, 8) | columns: ['case_id', 'modality', 'body_part', 'study_description', 'patient_age_band', 'patient_sex', 'template_content', 'dictation']
prompt tokens: max 1128 | mean 432


generating:   0%|          | 0/33 [00:00<?, ?it/s]

unfinished after retry: 0
status counts: {'ok': 132}
reports missing FINDINGS/IMPRESSION: 0
Rows: 132 | Columns: ['case_id', 'report']

First prediction:
 FINDINGS:
LIVER: Normal in size and attenuation. No focal hepatic lesion.

GALLBLADDER AND BILIARY TREE: Surgically absent gallbladder. Surgical clips are seen in the gallbladder fossa. No biliary ductal dilatation.

PANCREAS: Normal in size and contour.

SPLEEN: Small splenic calcification is noted. The spleen is otherwise normal.

ADRENAL GLANDS: Normal in size and morphology.

KIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.

URINARY BLADDER: Normal. No calculus or mass lesion.

REPRODUCTIVE: The prostate is normal in appearance.

MAJOR VESSELS: The abdominal aorta is normal in caliber.

PERITONEUM: No ascites. No pneumoperitoneum.

ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.

LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.

STOMACH AND BOWEL: Appendix is

In [16]:
m = df_model.reset_index(drop=True)
b = df_copy.reset_index(drop=True)

summary = pd.DataFrame({
    "copy-template": [b.findings_score.mean(), b.impression_score.mean(), b.res.mean(), b.res.median(), b.res.quantile(0.9)],
    "fine-tuned":    [m.findings_score.mean(), m.impression_score.mean(), m.res.mean(), m.res.median(), m.res.quantile(0.9)],
}, index=["FINDINGS mean", "IMPRESSION mean", "RES mean", "RES median", "RES p90"])
summary["improvement"] = summary["copy-template"] - summary["fine-tuned"]          # positive = model better
summary["improvement %"] = 100 * summary["improvement"] / summary["copy-template"]
print("LOWER IS BETTER\n", summary.round(4).to_string())

delta = b.res - m.res                                                              # positive = model better
print(f"\nmodel better: {(delta > 0.01).sum()} | about equal (|diff| <= 0.01): {(delta.abs() <= 0.01).sum()} | worse: {(delta < -0.01).sum()}  (of {len(m)})")
print("status counts:", pd.Series(val_status).value_counts().to_dict())

# ---- breakdown by modality and body part ----
bd = pred_df[["modality", "body_part"]].copy()
bd["model"], bd["copy"] = m.res.values, b.res.values
for col in ["modality", "body_part"]:
    g = bd.groupby(col).agg(n=("model", "size"), model=("model", "mean"), copy=("copy", "mean"))
    g["gain"] = g["copy"] - g["model"]
    print(f"\nBy {col} (n >= 3, sorted by gain):")
    print(g[g["n"] >= 3].sort_values("gain").round(3).to_string())

# ---- length sanity: too short = missing content, too long = rambling/repeats ----
ref_w = pred_df["reference_report"].str.split().str.len()
pred_w = pred_df["predicted_report"].str.split().str.len()
print("\npredicted/reference word-count ratio:\n", (pred_w / ref_w).describe().round(2).to_string())

# ---- cases where the model is worse than copying the template ----
wi = delta[delta < -0.01].sort_values().index
print("\nWorse than copy-template:")
print(pd.DataFrame({"case_id": m.case_id[wi], "model": m.res[wi], "copy": b.res[wi],
                    "status": pred_df["status"][wi]}).round(3).to_string(index=False))

LOWER IS BETTER
                  copy-template  fine-tuned  improvement  improvement %
FINDINGS mean           0.5862      0.2564       0.3299        56.2707
IMPRESSION mean         0.7342      0.3444       0.3898        53.0879
RES mean                0.6380      0.2872       0.3508        54.9888
RES median              0.7357      0.2778       0.4579        62.2422
RES p90                 0.8706      0.5572       0.3134        35.9963

model better: 81 | about equal (|diff| <= 0.01): 8 | worse: 6  (of 95)
status counts: {'ok': 94, 'fallback_template_impression': 1}

By modality (n >= 3, sorted by gain):
           n  model   copy   gain
modality                         
XRAY      64  0.273  0.598  0.326
CT        14  0.349  0.734  0.385
MRI       15  0.287  0.745  0.457

By body_part (n >= 3, sorted by gain):
                 n  model   copy   gain
body_part                              
Head             4  0.288  0.442  0.154
Ankle            3  0.314  0.517  0.203
Foot           

In [18]:
def wc(s): return len(str(s).split())

# ---- validation: prediction length vs reference and vs template ----
v = pred_df.copy()
v["ref_w"]  = v["reference_report"].map(wc)
v["pred_w"] = v["predicted_report"].map(wc)
v["tpl_w"]  = v["template_content"].map(wc)
v["ratio_ref"] = v.pred_w / v.ref_w.clip(lower=1)
v["ratio_tpl"] = v.pred_w / v.tpl_w.clip(lower=1)
print("Longest predictions vs reference (validation):")
print(v.sort_values("ratio_ref", ascending=False)[["case_id", "status", "ref_w", "pred_w", "tpl_w", "ratio_ref", "res"]].head(5).round(2).to_string(index=False))

# ---- test: no references, so compare with the template length ----
t = pd.DataFrame({"case_id": test_df["case_id"],
                  "pred_w": [wc(x) for x in test_preds],
                  "tpl_w":  [wc(x) for x in test_df["template_content"]]})
t["ratio_tpl"] = t.pred_w / t.tpl_w.clip(lower=1)

print("\npred/template word ratio  (median, max)")
print("  validation:", round(v.ratio_tpl.median(), 2), round(v.ratio_tpl.max(), 2))
print("  test      :", round(t.ratio_tpl.median(), 2), round(t.ratio_tpl.max(), 2))
print("\nTest reports to eyeball (longest relative to template):")
print(t.sort_values("ratio_tpl", ascending=False).head(5).round(2).to_string(index=False))

Longest predictions vs reference (validation):
                             case_id                       status  ref_w  pred_w  tpl_w  ratio_ref  res
14862dc4-ba90-462d-9661-8f51a5a5fe1e fallback_template_impression    275    1284    136       4.67 0.97
831d8f42-8abf-40c9-a577-f626a6c2abaa                           ok     40      73     50       1.82 0.66
c211ab46-0a3f-48a4-8cc1-8cebd8158bcb                           ok     79     109     29       1.38 0.44
95daab1b-9b1f-4c23-b971-31d6ace7f254                           ok     70      89     70       1.27 0.34
a9d77e62-d2bb-4f36-9961-7d378b993e98                           ok    182     228     88       1.25 0.16

pred/template word ratio  (median, max)
  validation: 2.37 9.44
  test      : 2.41 7.27

Test reports to eyeball (longest relative to template):
                             case_id  pred_w  tpl_w  ratio_tpl
697eea8a-94bf-4e40-9b09-49e1762f5475     189     26       7.27
3ac0bcca-274c-4354-a235-203868739ca9     177     28      

In [19]:
n = len(m)

mean_res_gain = 100 * (b.res.mean() - m.res.mean()) / b.res.mean()

findings_gain = 100 * (
    b.findings_score.mean() - m.findings_score.mean()
) / b.findings_score.mean()

impression_gain = 100 * (
    b.impression_score.mean() - m.impression_score.mean()
) / b.impression_score.mean()

improved = (delta > 0.01).sum()
equal = (delta.abs() <= 0.01).sum()
worse = (delta < -0.01).sum()

print(f"Test cases: {n}")
print(f"Mean RES reduction: {mean_res_gain:.2f}%")
print(f"Findings score reduction: {findings_gain:.2f}%")
print(f"Impression score reduction: {impression_gain:.2f}%")
print(f"Model better: {improved}/{n} ({100*improved/n:.1f}%)")
print(f"Approximately equal: {equal}/{n} ({100*equal/n:.1f}%)")
print(f"Model worse: {worse}/{n} ({100*worse/n:.1f}%)")

Test cases: 95
Mean RES reduction: 54.99%
Findings score reduction: 56.27%
Impression score reduction: 53.09%
Model better: 81/95 (85.3%)
Approximately equal: 8/95 (8.4%)
Model worse: 6/95 (6.3%)
